In [ ]:
# import requests
# from datetime import datetime


# url = "https://ai-benchmark.com/data/results_phones.htm"
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
# }

# response = requests.get(url, headers=headers)
# response.raise_for_status()

# # Save raw HTML with current timestamp
# timestamp = datetime.now().strftime("ai-benchmark-%Y-%m-%d-%H-%M-%S")
# html_filename = f"{timestamp}.html"

# with open(html_filename, "w", encoding="utf-8") as file:
#     file.write(response.text)
# print(f"Raw HTML saved to {html_filename}\n")

Raw HTML saved to ai-benchmark-2026-04-20-15-47-53.html



In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

file_path = "ai-benchmark-2026-04-20-15-47-53.html"
df = pd.read_html(file_path)[0]

phone_col = df.columns[df.iloc[1].str.contains(r'Phone\s+Model', regex=True, na=False)][0]
chipset_col = df.columns[df.iloc[1] == 'Chipset'].tolist()[0]

gpt2_cpu_f_col = None
for col in df.columns:
    if str(df.iloc[1, col]).strip() == 'GPT-2' and str(df.iloc[2, col]).strip() == 'CPU-F':
        gpt2_cpu_f_col = col
        break

if gpt2_cpu_f_col is not None:
    final_df = df.iloc[5:][[phone_col, chipset_col, gpt2_cpu_f_col]].copy()
    final_df.columns = ['device_name', 'chipset_name', 'compute_latency_ms']
    
    final_df['compute_latency_ms'] = pd.to_numeric(final_df['compute_latency_ms'], errors='coerce')
    final_df = final_df.dropna(subset=['compute_latency_ms']).reset_index(drop=True)
    
    # Plot before cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=final_df['compute_latency_ms'])
    plt.title('Before Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(final_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('Before Cleaning: Distribution')
    plt.savefig('chart_before_cleaning.png')
    plt.close()
    
    # 1. Filter out 0 ms
    cleaned_df = final_df[final_df['compute_latency_ms'] > 0].copy()
    
    # 2. Filter out upper outliers using IQR
    Q1 = cleaned_df['compute_latency_ms'].quantile(0.25)
    Q3 = cleaned_df['compute_latency_ms'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    
    cleaned_df = cleaned_df[cleaned_df['compute_latency_ms'] <= upper_bound].reset_index(drop=True)
    
    # Plot after cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=cleaned_df['compute_latency_ms'])
    plt.title('After Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(cleaned_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('After Cleaning: Distribution')
    plt.savefig('chart_after_cleaning.png')
    plt.close()
    
    print(f"Original Count: {len(final_df)}")
    print(f"Cleaned Count: {len(cleaned_df)}")
    print(f"Removed {len(final_df) - len(cleaned_df)} outliers/zeros.")
    
    # Save the cleaned dataframe
    cleaned_df.to_json("trace.json", orient="records", indent=4)
else:
    print("Could not locate the GPT-2 CPU-F column.")

Original Count: 633
Cleaned Count: 568
Removed 65 outliers/zeros.
